# Cost, latency and caching: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads the small embedding model all-MiniLM-L6-v2 (about 90 MB) and a tokenizer file for tiktoken; it needs internet but no GPU and no key. No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install sentence-transformers==6.3.0 numpy==2.5.2 tiktoken==0.14.0 langchain-core==1.8.4 langchain-classic==1.1.0

## Embedding cache and idempotency

A hash-keyed cache in front of a real embedding model, counting the texts it encodes.

In [ ]:
import hashlib
import json

import numpy as np
from sentence_transformers import SentenceTransformer

NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(NAME)
cache, encoded = {}, 0

def embed(texts):
    global encoded
    keys = [hashlib.sha256(json.dumps([NAME, True, t]).encode()).hexdigest() for t in texts]
    missing = {k: t for k, t in zip(keys, texts) if k not in cache}
    if missing:
        vectors = model.encode(list(missing.values()), normalize_embeddings=True)
        cache.update(zip(missing, vectors))
        encoded += len(missing)
    return np.array([cache[k] for k in keys])

texts = ["The Business plan commits to 99.9 percent monthly uptime.",
         "Deleted tickets stay in the trash for 30 days.",
         "The Business plan commits to 99.9 percent monthly uptime."]
first = embed(texts)
print("shape:", first.shape, "| texts asked:", len(texts), "| encoded:", encoded)
texts[1] = "Deleted tickets stay in the trash for 45 days."
second = embed(texts)
print("after one edit, encoded in total:", encoded)
fresh = model.encode(texts, normalize_embeddings=True)
print("cached vectors match fresh ones:", bool(np.allclose(second, fresh, atol=1e-5)))

shape: (3, 384) | texts asked: 3 | encoded: 2
after one edit, encoded in total: 3
cached vectors match fresh ones: True


## LLM token budgeting

Words and tokens of five short strings, counted with the tiktoken library.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
samples = ["How many days of leave do I get?",
           "uncharacteristically",
           "faq-export@v2",
           "9876543210123",
           "x = df.groupby('tenant')['tokens'].sum()"]

for text in samples:
    ids = enc.encode(text)
    print(f"{len(text.split())} words, {len(ids):2d} tokens: {text}")
print([enc.decode([i]) for i in enc.encode("faq-export@v2")])

8 words,  9 tokens: How many days of leave do I get?
1 words,  3 tokens: uncharacteristically
1 words,  5 tokens: faq-export@v2
1 words,  5 tokens: 9876543210123
3 words, 11 tokens: x = df.groupby('tenant')['tokens'].sum()
['faq', '-export', '@', 'v', '2']


## Streaming vs batch

Reading a streamed answer piece by piece with a stand-in model from langchain-core.

In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

# a stand-in that replays a fixed answer; no real model is called
model = GenericFakeChatModel(messages=iter([AIMessage(content="Deleted tickets stay 30 days. [faq-trash]")]))

pieces = []
for chunk in model.stream("How long do deleted tickets stay in the trash?"):
    pieces.append(chunk.content)
    print(len(pieces), repr(chunk.content))

print("joined:", "".join(pieces))
print("pieces:", len(pieces))

1 'Deleted'
2 ' '
3 'tickets'
4 ' '
5 'stay'
6 ' '
7 '30'
8 ' '
9 'days.'
10 ' '
11 '[faq-trash]'
joined: Deleted tickets stay 30 days. [faq-trash]
pieces: 11


## Notebook: caching end to end

Counting what is encoded with CacheBackedEmbeddings and an in-memory store.

In [ ]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import InMemoryByteStore
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

class Counting(Embeddings):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.encoded = 0                       # texts really sent to the model

    def embed_documents(self, texts):
        self.encoded += len(texts)
        return self.model.encode(list(texts)).tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]

def run(label, emb, inner):
    before = inner.encoded
    emb.embed_documents(["Trash is kept 30 days.", "Backups are kept 35 days."])
    a = inner.encoded - before
    emb.embed_documents(["Trash is kept 30 days.", "Exports last 24 hours."])
    b = inner.encoded - before - a
    emb.embed_query("how long is the trash kept")
    emb.embed_query("how long is the trash kept")
    c = inner.encoded - before - a - b
    print(f"{label}: first batch {a}, second batch {b}, same query twice {c}")

inner = Counting()
plain = CacheBackedEmbeddings.from_bytes_store(inner, InMemoryByteStore(), namespace="minilm-v1", key_encoder="sha256")
run("documents cached only", plain, inner)
both = CacheBackedEmbeddings.from_bytes_store(inner, InMemoryByteStore(), namespace="minilm-v1", key_encoder="sha256",
                                              query_embedding_cache=True)
run("queries cached too   ", both, inner)

documents cached only: first batch 2, second batch 1, same query twice 2
queries cached too   : first batch 2, second batch 1, same query twice 1
